# Cost functions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/04_costs.ipynb)

A `CostFunction` defines the running cost density $g(x, u, t)$ and terminal cost $h(x, t)$ of the continuous-time objective

$$J = \int_{t_0}^{t_0 + t_f} e^{-\rho(t - t_0)}\, g(x(t), u(t), t)\, dt + e^{-\rho t_f}\, h(x(t_0 + t_f), t_0 + t_f),$$

while a `StepCost` (`SampledCost`) defines the per-tick stage cost $c(x_k, u_k, k)$ and terminal cost $h(x_N, N)$ of a discrete-time problem.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. `CostFunction`: the running cost `g` and the terminal cost `h`

A `CostFunction` is the integrand and the terminal term of the objective

$$J = \int_{t_0}^{t_f} e^{-\rho (t - t_0)}\, g(x, u, t)\, dt + e^{-\rho (t_f - t_0)}\, h(x(t_f), t_f).$$

The horizon and the discount rate $\rho$ are the planning problem's; the cost is `g` and `h` only, so one cost serves many problems.

| Call | Returns | Meaning |
| --- | --- | --- |
| `cost.params` | `dict` | The cost's numbers |
| `cost.id` | `str \| None` | The cost's key in a sum (`None`: its class name) |
| `cost.g(x, u, t=0.0, params=None)` | scalar | Running cost $g(x, u, t)$; the one method a subclass writes |
| `cost.h(x, t=0.0, params=None)` | scalar | Terminal cost $h(x, t)$, zero unless the subclass charges one |
| `cost.integrate(traj, *, discount_rate=0.0, params=None, t0=0.0)` | `(dJ, running)`, each `array (N,)` | The discounted integrand $e^{-\rho (t_k - t_0)} g_k$ at each sample and its trapezoidal integral from $t_0$; `h` is not included |
| `cost.g_max(X, U, params=None)` | `float` | The largest `g` over the boxes `X` and `U` (two `BoxSet`s): exact on `QuadraticCost` and `TimeCost`, a sampled estimate on the base class |
| `cost.h_max(X, params=None)` | `float` | The largest `h` over the box `X`, likewise |
| `c1 + c2` | `SumCost` | The sum of two costs |
| `w * cost` | `ScaledCost` | The cost scaled by the number `w` |

The library costs:

| Class | Cost | `params` |
| --- | --- | --- |
| `QuadraticCost(Q, R, S=None, x_bar=None, u_bar=None)` | $g = (x - \bar x)^\top Q\,(x - \bar x) + (u - \bar u)^\top R\,(u - \bar u)$, $h = (x - \bar x)^\top S\,(x - \bar x)$; `S`, `x_bar`, `u_bar` default to zero | `{"Q", "R", "S", "x_bar", "u_bar"}`, with the views `cost.Q`, `cost.R`, `cost.S`, `cost.x_bar`, `cost.u_bar` |
| `QuadraticCost.from_system(sys, *, Q=None, R=None, S=None, x_bar=None, u_bar=None)` | The same, sized by the system: identity `Q` and `R`, zero `S`, and the system's nominal state and input as the target | |
| `TimeCost(x_bar, radius=1e-3)`, `TimeCost.from_system(sys, *, x_bar=None, radius=1e-3)` | Minimum time: $g = 1$, except $g = 0$ within `radius` of `x_bar`; $h = 0$ | `{"x_bar", "radius"}` |
| `SumCost(terms)`, `SumCost.of(*costs, **named)` | $\sum_i J_i$; `total.terms` is `{term_id: cost}` | `{term_id: term.params}` |
| `ScaledCost(cost, weight)` | $w\,J_0$ | `{"weight", "cost"}` |
| `FieldCost(field, weight=1.0, shaping=None)`, `FieldTerminalCost(field, weight=1.0, shaping=None)` | $g$ or $h$ from a field, built by `V.as_cost()`, `V.as_terminal_cost()` and `S.as_cost()` ([`03_sets_fields_distributions`](03_sets_fields_distributions.ipynb)) | `{"weight", "field"}`, and `"shaping"` when shaped |

In [ ]:
import numpy as np

from minilink import Pendulum, QuadraticCost

plant = Pendulum()
cost = QuadraticCost.from_system(
    plant,
    Q=np.diag([10.0, 1.0]),
    R=0.1 * np.eye(1),
    S=5.0 * np.eye(2),
    x_bar=np.array([np.pi, 0.0]),
)
cost

In [ ]:
x = np.array([0.5, 0.0])
u = np.array([1.0])

cost.g(x, u, t=0.0), cost.h(x, t=5.0)

In [ ]:
# The largest g and h over the plant's state and input boxes
cost.g_max(plant.state.box, plant.input_box.box), cost.h_max(plant.state.box)

## 2. The cost of a `Trajectory`

| Call | Returns | Meaning |
| --- | --- | --- |
| `cost.integrate(traj, *, discount_rate=0.0, params=None, t0=0.0)` | `(dJ, running)` | The running cost along the samples (above) |
| `traj.with_cost(cost, *, discount_rate=0.0, params=None, t0=0.0)` | `Trajectory` | A copy with the signals `"dJ"` and `"J"`, each `(1, N)`; the discounted `h` is added at the last sample of `"J"`, so its last value is the run's $J$ |
| `sys.compute_cost(cost, *, of=None, traj=None)` | `float` | $J$ of `traj` (default `sys.traj`); on a diagram, `of="plant"` scores that block's own run |
| `sys.plot_cost(cost, *, of=None, traj=None, renderer="matplotlib", show=True)` | `PlotResult` | Plot `"dJ"` and `"J"` over time |

In [ ]:
plant.x0 = np.array([1.0, 0.0])
traj = plant.compute_trajectory(tf=5.0)

dJ, running = cost.integrate(traj)
dJ.shape, running.shape, running[-1]

In [ ]:
scored_traj = traj.with_cost(cost)
scored_traj.signal_names, scored_traj.signal("J")[0, -1], plant.compute_cost(cost, traj=traj)

In [ ]:
plant.plot_cost(cost, traj=traj);

## 3. Cost algebra, soft constraints, and `TimeCost`

`+` builds a `SumCost` and `*` a `ScaledCost`. In `a + b` each term is keyed by its `id`, else its class name (a scaled cost takes the key of the cost it scales); `SumCost.of(tracking=..., barrier=...)` names the terms. `total.params` is the live view `{term_id: term.params}`.

In [ ]:
from minilink import BoxSet, TimeCost

wall = BoxSet([-2.0, -3.0], [2.0, 3.0]).as_cost()
total = cost + 50.0 * wall
total

In [ ]:
from minilink.core import SumCost

named_total = SumCost.of(tracking=cost, barrier=50.0 * wall)
list(named_total.terms), named_total.params["barrier"], named_total.g(np.array([2.5, 0.0]), u)

In [ ]:
min_time = TimeCost.from_system(plant, x_bar=np.array([np.pi, 0.0]), radius=0.05)
(
    min_time.g(np.array([0.0, 0.0]), u),
    min_time.g(np.array([np.pi, 0.0]), u),
    min_time.g_max(plant.state.box, plant.input_box.box),
)

### Writing a `CostFunction`

A cost subclasses `CostFunction`, sets `self.params`, and writes `g` (and `h` when it charges a terminal cost) on `array_module(x)`, so the same equation runs on NumPy and traces on JAX:

In [ ]:
import jax.numpy as jnp

from minilink import CostFunction
from minilink.core.backends import array_module


class SwingUpCost(CostFunction):
    def __init__(self):
        self.params = {"w_dtheta": 0.1, "w_u": 0.001}

    def g(self, x, u, t=0.0, params=None):
        params = self.params if params is None else params
        xp = array_module(x)

        w_dtheta = params["w_dtheta"]
        w_u = params["w_u"]

        theta = x[0]
        dtheta = x[1]
        tau = u[0]

        # e = θ − π, wrapped to (−π, π]
        e = xp.arctan2(xp.sin(theta - xp.pi), xp.cos(theta - xp.pi))
        g = e**2 + w_dtheta * dtheta**2 + w_u * tau**2

        return g


swing_up = SwingUpCost()
(
    swing_up.g(np.array([-3.1, 0.0]), u),
    swing_up.g(jnp.array([3.1, 0.0]), jnp.array([1.0])),
    swing_up.h(x),
)

## 4. `StepCost` and `SampledCost`: costs on a clock

A `StepCost` is the cost of a step problem, a stage cost $c(x_k, u_k, k)$ and a terminal cost $h(x_N, N)$ of

$$J = \sum_{k=0}^{N-1} \gamma^k\, c(x_k, u_k, k) + \gamma^N\, h(x_N, N).$$

| Call | Returns | Meaning |
| --- | --- | --- |
| `step_cost.c(x, u, k=0, params=None)` | scalar | Stage cost of tick $k$ |
| `step_cost.h(x, k=0, params=None)` | scalar | Terminal cost at tick $k$ |

`SampledCost(cost, plant, dt, *, solver="rk4", substeps=1, discount_rate=0.0, t0=0.0)` is the `StepCost` of a continuous cost on a plant sampled at `dt`: it integrates $g$ beside the plant over one period, the input held. Its `params` are the continuous cost's. `problem.discretize(dt)` builds one ([`05_problem`](05_problem.ipynb)).

| Call | Returns | Meaning |
| --- | --- | --- |
| `sampled.c(x, u, k=0, params=None)` | scalar | $c_k = \int_{t_k}^{t_k + \Delta t} e^{-\rho(\tau - t_k)}\, g(x(\tau), u, \tau)\, d\tau$, with $t_k = t_0 + k\,\Delta t$ |
| `sampled.h(x, k=0, params=None)` | scalar | $h(x, t_k)$ |
| `sampled.stage_at(x, u, t_k, params=None, plant_params=None, discount_rate=None)` | `(x_next, c_k)` | The plant's step and its stage cost in one integration, from the absolute time `t_k` |
| `sampled.stage_cost_at(x, u, t_k, params=None, plant_params=None, discount_rate=None)` | scalar | $c_k$ alone |
| `sampled.terminal_at(x, t, params=None)` | scalar | $h(x, t)$ at the absolute time `t` |

In [ ]:
from minilink.core import SampledCost

sampled_cost = SampledCost(cost, plant, dt=0.05)
c_0 = sampled_cost.c(x, u, k=0)
h_0 = sampled_cost.h(x, k=0)
x_next, c_fused = sampled_cost.stage_at(x, u, t_k=0.0)

c_0, h_0, x_next, c_fused

Next, [`05_problem`](05_problem.ipynb): `PlanningProblem` and `StepPlanningProblem`.